In [1]:
from transformers import BertForSequenceClassification, BertTokenizer
from sklearn.ensemble import StackingClassifier
from sklearn.metrics import classification_report
import tensorflow as tf
from datasets import load_dataset

c:\Users\Aishu\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name = 'bert-base-multilingual-cased'
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2)
tokenizer = BertTokenizer.from_pretrained(model_name)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
import pandas as pd

# Load dataset
df = pd.read_csv("data_generation\dataset.csv")  
print(df.head())

                                            headline  label
0    Kal ka plan final hai na  Time pe pahuch jaana       0
1  Tune suna Sharma ji ka beta first aaya exams m...      0
2  Chalo beach pe chalte hain weekend par maza aa...      0
3    Tu itna late kyun ho gaya aaj  Traffic tha kya       0
4  Woh new series dekhi kya Netflix par  Bahut ac...      0


In [4]:
from sklearn.model_selection import train_test_split


X = df['headline']  
y = df['label'] 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training size:", len(X_train))
print("Testing size:", len(X_test))

Training size: 3281
Testing size: 821


In [5]:
from datasets import load_dataset
dataset = load_dataset("csv", data_files="data_generation\dataset.csv")

In [6]:
def tokenize_data(texts, labels, tokenizer, max_length=128):
    encodings = tokenizer(
        list(texts), 
        max_length=max_length, 
        truncation=True, 
        padding="max_length", 
        return_tensors="pt"
    )
    return encodings, labels


In [7]:
train_encodings, train_labels = tokenize_data(X_train, y_train, tokenizer)

test_encodings, test_labels = tokenize_data(X_test, y_test, tokenizer)

In [15]:
print(train_encodings)
print(train_labels)

{'input_ids': tensor([[  101, 11186,   887,  ...,     0,     0,     0],
        [  101, 13080, 13080,  ...,     0,     0,     0],
        [  101, 77826, 28546,  ...,     0,     0,     0],
        ...,
        [  101,   165,   158,  ...,     0,     0,     0],
        [  101,   851, 28546,  ...,     0,     0,     0],
        [  101, 65272, 10113,  ...,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])}
1178    0
2954    0
3859    0
67      0
2877    0
       ..
3444    0
466     0
3092    1
3772    0
860     1
Name: label, Length: 3281, dtype: int64

In [9]:
from transformers import TrainingArguments,Trainer

In [10]:
training_args = TrainingArguments(
    output_dir="./results",        # Directory to save the model
    evaluation_strategy="epoch",  # Evaluate after each epoch
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=1,
    load_best_model_at_end=True,
)

c:\Users\Aishu\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [11]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch

In [12]:

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}


In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset={'data': train_encodings,'label':train_labels},
    eval_dataset={'data': test_encodings,'label': test_labels},
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

C:\Users\Aishu\AppData\Local\Temp\ipykernel_14012\1649834084.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [14]:
trainer.train()

  0%|          | 0/3 [00:00<?, ?it/s]

KeyError: 0